# Simple CNN testing

In [1]:
import torch
import wandb
import time
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from src.cnn_utils import get_dataset

In [2]:
print(wandb)
print(wandb.__file__)

<module 'wandb' from 'c:\\Users\\lucas\\Documents\\MSE\\4_Semester\\FTP_DeLearn\\venv_DeLearn\\Lib\\site-packages\\wandb\\__init__.py'>
c:\Users\lucas\Documents\MSE\4_Semester\FTP_DeLearn\venv_DeLearn\Lib\site-packages\wandb\__init__.py


In [3]:
# Check for GPU
device = None
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else: 
    device = torch.device("cpu")

print(device)

cuda


In [4]:
train_dataset, val_dataset = get_dataset()

In [5]:
# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

Training dataset size: 24000
Validation dataset size: 6000


### Create Training loop for models

In [6]:
def train_eval(model, optimizer, nepochs, batch_size, training_data, validation_data, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN',run_name=None, use_wandb=True):
    """
    Train and evaluate a model.
    Logs train/validation loss and accuracy to Weights & Biases if use_wandb=True.
    """
    cost_hist = []
    cost_hist_val = []
    acc_hist = []
    acc_hist_val = []

    model = model.to(device) # <-- move model to device (GPU or CPU)
    cost_ce = torch.nn.CrossEntropyLoss().to(device)
    
    train_loader = DataLoader(training_data, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(validation_data, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    
    if use_wandb:
        wandb.init(
            entity=entity,
            project=project,
            name=run_name,
            settings=wandb.Settings(init_timeout=300),
            config={
                "epochs": nepochs,
                "batch_size": batch_size,
                "optimizer": optimizer.__class__.__name__,
                "loss": "CrossEntropyLoss",
                "device": str(device),
                "model": model.__class__.__name__
            }
        )
        wandb.watch(model, log="all", log_freq=100)

    for epoch in range(nepochs):
        start = time.perf_counter()

        model.train()
        size = len(train_loader.dataset)
        nbatches = len(train_loader)
        cost, acc = 0.0, 0.0
        for batch, (X, Y) in enumerate(train_loader):
            X,Y = X.to(device),Y.to(device)
            pred = model(X)
            loss = cost_ce(pred, Y)
            cost += loss.item()
            acc += (pred.argmax(dim=1) == Y).type(torch.float).sum().item()

            # gradient, parameter update
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        cost /= nbatches
        acc /= size

        model.eval()
        size_val = len(val_loader.dataset)
        nbatches_val = len(val_loader)
        cost_val, acc_val = 0.0, 0.0     

        with torch.no_grad():
            for X, Y in val_loader:
                X,Y = X.to(device),Y.to(device)
                pred = model(X)
                cost_val += cost_ce(pred, Y).item()
                acc_val += (pred.argmax(dim=1) == Y).type(torch.float).sum().item()

        cost_val /= nbatches_val
        acc_val /= size_val

        end = time.perf_counter()
        epoch_time = end - start

        print(f"Epoch {epoch}: Train cost: {round(cost, 4)}, accuracy: {round(acc, 4)}, Validation cost: {round(cost_val, 4)}, accuracy: {round(acc_val, 4)} (Time: {round(epoch_time, 1)} seconds)")

        cost_hist.append(cost)
        cost_hist_val.append(cost_val)
        acc_hist.append(acc)
        acc_hist_val.append(acc_val)

        if use_wandb:
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": cost,
                "train_accuracy": acc,
                "val_loss": cost_val,
                "val_accuracy": acc_val,
                "lr": optimizer.param_groups[0]['lr'],
                "epoch_time": epoch_time
            })

    if use_wandb:
        wandb.finish()

    return cost_hist, cost_hist_val, acc_hist, acc_hist_val

### Creating shallow CNN-model

In [7]:
# creat a simple model with one convolutional layer and two fully connected layers

class first_model(nn.Module):
    
    def __init__(self, units=128):
        super(first_model, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Conv with 32 filters, kernel size 3x3, padding 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(112*112*32,units),
            nn.ReLU(),
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [8]:
# create an model and its summary

model = first_model(128) # no need for to(device), it breaks when running on Apple mps chip
from torchsummary import summary
summary(model, (3,224,224),device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
           Flatten-4               [-1, 401408]               0
            Linear-5                  [-1, 128]      51,380,352
              ReLU-6                  [-1, 128]               0
            Linear-7                   [-1, 10]           1,290
Total params: 51,382,538
Trainable params: 51,382,538
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 30.63
Params size (MB): 196.01
Estimated Total Size (MB): 227.21
----------------------------------------------------------------


Initiate Training

In [9]:
batch_size = 64
nepochs = 50
lr = 0.001
units = 128

model = first_model(units)
optimizer = torch.optim.SGD(params=model.parameters(), lr = lr)
cost_train_sgd, cost_valid_sgd, acc_train_sgd, acc_valid_sgd = train_eval(model, optimizer, nepochs, batch_size, train_dataset, val_dataset, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='CNN-testing_bs64_e30_lr0.001_u128_1l_no-reg', use_wandb=True)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lucas\_netrc.
wandb: Currently logged in as: lucas-j-keller98 (MSE_DeLearn_SPR26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 0: Train cost: 2.2489, accuracy: 0.1636, Validation cost: 2.2132, accuracy: 0.2133 (Time: 31.1288 seconds)
Epoch 1: Train cost: 2.1767, accuracy: 0.2215, Validation cost: 2.1713, accuracy: 0.2145 (Time: 30.9657 seconds)
Epoch 2: Train cost: 2.1237, accuracy: 0.2533, Validation cost: 2.1077, accuracy: 0.2812 (Time: 29.2917 seconds)
Epoch 3: Train cost: 2.0723, accuracy: 0.2845, Validation cost: 2.0611, accuracy: 0.2913 (Time: 29.2583 seconds)
Epoch 4: Train cost: 2.0185, accuracy: 0.3065, Validation cost: 2.0244, accuracy: 0.3118 (Time: 29.1928 seconds)
Epoch 5: Train cost: 1.9642, accuracy: 0.3252, Validation cost: 1.9604, accuracy: 0.3362 (Time: 29.3365 seconds)
Epoch 6: Train cost: 1.9164, accuracy: 0.342, Validation cost: 1.9475, accuracy: 0.3243 (Time: 29.2958 seconds)
Epoch 7: Train cost: 1.8756, accuracy: 0.3522, Validation cost: 1.8855, accuracy: 0.339 (Time: 29.6718 seconds)
Epoch 8: Train cost: 1.8402, accuracy: 0.365, Validation cost: 1.8645, accuracy: 0.3542 (Time: 29.

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
epoch_time,█▇▁▁▁▁▃▂▂▃▁▂▂▂▁▂▂▃▂▁▂▂▂▁▂▁▂▂▂▂▂▂▁▂▄█▇██▇
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▂▂▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇███
train_loss,██▇▇▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
val_accuracy,▁▁▃▄▅▅▆▅▆▅▆▆▆▆▇▇▇▇▇▇█▇▇█▇▇█▇██▇███████▇█
val_loss,██▇▆▅▄▄▄▃▄▃▃▃▃▂▂▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▁
epoch,50
epoch_time,31.10418
lr,0.001
train_accuracy,0.61158


This first model is heavily in a overfitting regime.\
Really bad vallidation accuracy and loss.\
The Result was kind of expected looking at the amount of parameters that are estimted during training (+40 Mio).\
Therefore we tried to construct a shallow model that performs better.
Possibilities to improve the model:
- heavier downsampling before passing into a fully connected layer
    --> adding more layers before fully connectde layers (Conv2d --> ReLu --> MaxPool2d)
- reduce/ make the dense layer smaller (less units)
- add regularization:
    - Dropout
    - better optimizer
    - early stopping
    - etc.


In [17]:
# creat a simple model with one convolutional layer and two fully connected layers

class improved_model(nn.Module):
    
    def __init__(self, units=128, drop=0.5):
        super(improved_model, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'), # Conv with 32 filters, kernel size 3x3 and padding 
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(112*112*32,units),
            nn.ReLU(),
            nn.Dropout(drop), # dropout rate of 0.5 as deafault value
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [18]:
model = improved_model(128, 0.5) # no need for to(device), it breaks when running on Apple mps chip
from torchsummary import summary
summary(model, (3,224,224),device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
           Flatten-4               [-1, 401408]               0
            Linear-5                  [-1, 128]      51,380,352
              ReLU-6                  [-1, 128]               0
           Dropout-7                  [-1, 128]               0
            Linear-8                   [-1, 10]           1,290
Total params: 51,382,538
Trainable params: 51,382,538
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 30.63
Params size (MB): 196.01
Estimated Total Size (MB): 227.21
----------------------------------------------------------------


In [ ]:
batch_size = 64
nepochs = 50
lr = 1
units = 128
wd = 1e-4 # default weight decay value for Adam optimizer, can be tuned as a hyperparameter

model = improved_model(units)
optimizer = torch.optim.Adam(params=model.parameters(), lr = lr, weight_decay=wd)
cost_train_adam, cost_valid_adam, acc_train_adam, acc_valid_adam = train_eval(model, optimizer, nepochs, batch_size, train_dataset, val_dataset, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='CNN-testing_bs64_e50_lr1_u128_1l_adam_drop0.5', use_wandb=True)


Epoch 0: Train cost: 115199.3207, accuracy: 0.0971, Validation cost: 2.3169, accuracy: 0.1 (Time: 31.5 seconds)
Epoch 1: Train cost: 2.3641, accuracy: 0.1031, Validation cost: 2.3526, accuracy: 0.1 (Time: 32.3 seconds)
Epoch 2: Train cost: 2.3751, accuracy: 0.0958, Validation cost: 2.3341, accuracy: 0.1 (Time: 31.7 seconds)
Epoch 3: Train cost: 2.3737, accuracy: 0.1031, Validation cost: 2.3919, accuracy: 0.1 (Time: 31.6 seconds)
Epoch 4: Train cost: 2.3979, accuracy: 0.0976, Validation cost: 2.4851, accuracy: 0.1 (Time: 31.7 seconds)
Epoch 5: Train cost: 2.3712, accuracy: 0.099, Validation cost: 2.3419, accuracy: 0.1 (Time: 31.5 seconds)
Epoch 6: Train cost: 2.4178, accuracy: 0.0981, Validation cost: 2.3555, accuracy: 0.1 (Time: 31.4 seconds)
Epoch 7: Train cost: 2.3822, accuracy: 0.1, Validation cost: 2.4068, accuracy: 0.1 (Time: 31.5 seconds)
Epoch 8: Train cost: 2.4509, accuracy: 0.0978, Validation cost: 2.3618, accuracy: 0.1 (Time: 31.6 seconds)
Epoch 9: Train cost: 2.3662, accurac